# 05 LoRA persona DPO
Attach: repo zip dataset. Accelerator: GPU T4 x2 (one GPU is used). Internet: on.

In [ ]:
import glob, os, shutil, zipfile
matches = glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)
if not matches:
    for archive in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        zipfile.ZipFile(archive).extractall("/tmp/unzipped")
    matches = glob.glob("/tmp/unzipped/**/submission_adapter.py", recursive=True)
assert matches, "attach the repo zip as a Kaggle dataset"
REPO = "/tmp/repo"
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(os.path.dirname(matches[0]), REPO)
os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO
OUT = "/kaggle/working/outputs"
os.makedirs(OUT, exist_ok=True)
print(os.listdir(REPO))

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest peft trl
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "transformers", transformers.__version__, "datasets", datasets.__version__)
print("gpus", torch.cuda.device_count())

import peft, trl
print('peft', peft.__version__, 'trl', trl.__version__)

## Smoke test (4 steps)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python scripts/train_lora_dpo.py --output_dir /tmp/smoke_lora --epochs 1 --max_steps 4

## Full run

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python scripts/train_lora_dpo.py --output_dir {OUT}/lora_dpo

## Generation on the style evaluation prompts

In [ ]:
PERSONA = "eval_sets/persona_style_eval.json"
BASE = "HuggingFaceTB/SmolLM2-360M-Instruct"
GEN_ARGS = "--max_new_tokens 256 --batch_size 8 --chat_template"
os.makedirs(f"{OUT}/gen", exist_ok=True)
os.makedirs(f"{OUT}/eval", exist_ok=True)
!python scripts/generate.py --model {BASE} --prompts_path {PERSONA} --output_path {OUT}/gen/smollm2_instruct_persona.json --generator smollm2_instruct --dataset persona_style_eval {GEN_ARGS}
!python scripts/generate.py --model {BASE} --adapter_path {OUT}/lora_dpo/adapter --prompts_path {PERSONA} --output_path {OUT}/gen/lora_dpo_persona.json --generator lora_dpo --dataset persona_style_eval {GEN_ARGS}

In [ ]:
import json
base = json.load(open(f"{OUT}/gen/smollm2_instruct_persona.json"))
lora = json.load(open(f"{OUT}/gen/lora_dpo_persona.json"))
for b, l in list(zip(base, lora))[:3]:
    print(b["instruction"], "\nBASE:", b["output"], "\nLORA:", l["output"], "\n")

In [ ]:
!python scripts/build_pairs.py --subject_path {OUT}/gen/lora_dpo_persona.json --baseline_path {OUT}/gen/smollm2_instruct_persona.json --subject_name lora_dpo --baseline_name smollm2_instruct --prompts_path {PERSONA} --kind persona --output_path {OUT}/eval/lora_vs_base_persona_pairs.jsonl
!cd /kaggle/working && zip -rq lora_outputs.zip outputs/lora_dpo outputs/gen outputs/eval